# MAIA — Producción Final (Gemma 4 E4B → Maia)

**Pipeline completo: SFT + DPO → calitosaa/Maia**

- Base: `unsloth/gemma-4-E4B-it` (4B params, 128K ctx, multimodal)
- Dataset: 133K+ ejemplos + identidad Maia
- Método: QLoRA 4-bit + LoRA r=16 + SFT + DPO
- GPU: T4 16GB (Google Colab Free)
- Output: `calitosaa/Maia` (16-bit) + `calitosaa/Maia-GGUF` (Q4_K_M)
- Duración: ~14-18h total

In [ ]:
import os, torch
from getpass import getpass

# === CONFIG ===
BASE_MODEL      = 'unsloth/gemma-4-E4B-it'
HF_REPO         = 'calitosaa/Maia'
HF_REPO_GGUF    = 'calitosaa/Maia-GGUF'
MAX_SEQ_TRAIN   = 8192
INFERENCE_CTX   = 131072   # 128K
SFT_EPOCHS      = 1
SFT_LR          = 2e-4
SFT_BATCH       = 2
SFT_GRAD_ACCUM  = 4
DPO_EPOCHS      = 1
DPO_LR          = 5e-6
DPO_BATCH       = 2
DPO_BETA        = 0.1

# HF Token desde Colab Secrets (key: HF_TOKEN)
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    HF_TOKEN = getpass('HuggingFace token (read+write): ')
    os.environ['HF_TOKEN'] = HF_TOKEN

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'Token OK: {HF_TOKEN[:8]}...')

In [ ]:
# Instalar dependencias
!pip install -q -U unsloth
!pip install -q -U 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q -U datasets trl peft accelerate bitsandbytes huggingface_hub sentencepiece protobuf

from huggingface_hub import login, HfApi
login(token=HF_TOKEN, add_to_git_credential=True)
api = HfApi(token=HF_TOKEN)
USERNAME = api.whoami()['name']
print(f'Logged in as: {USERNAME}')
print('Deps OK')

In [ ]:
# Clonar repo Maia y ensamblar dataset
!git clone --depth 1 https://github.com/calitosaa/Maia /content/maia_repo
import os, subprocess
os.chdir('/content/maia_repo/finetuning/output')

# Ensamblar partes del dataset base
!cat maia_gemma4_finetune.jsonl.part_* > maia_base.jsonl
base_lines = int(subprocess.run(['wc','-l','maia_base.jsonl'], capture_output=True, text=True).stdout.split()[0])
print(f'Base dataset: {base_lines:,} ejemplos')

In [ ]:
# Generar datos de identidad de Maia
import json

IDENTITY = [{"messages": [{"role": "user", "content": "¿Quién eres?"}, {"role": "assistant", "content": "Soy Maia, un asistente de IA avanzado creado mediante fine-tuning de Gemma 4 E4B. Tengo 55+ agentes especializados para orquestación, razonamiento, código, visión, RAG y más. ¿En qué puedo ayudarte?"}]}, {"messages": [{"role": "user", "content": "What are you?"}, {"role": "assistant", "content": "I'm Maia, an advanced AI assistant built on Gemma 4 E4B with 128K context window. I have 55+ specialized agents for coding, reasoning, RAG, vision, security, and domain-specific tasks. How can I help you?"}]}, {"messages": [{"role": "user", "content": "¿Qué puedes hacer?"}, {"role": "assistant", "content": "Puedo ayudarte con:\n- **Código**: depuración, optimización, generación (Python, JS, etc.)\n- **Razonamiento**: problemas complejos con Chain-of-Thought\n- **RAG**: búsqueda y recuperación de información\n- **Visión**: análisis de imágenes, OCR, diagramas\n- **Seguridad**: detección de vulnerabilidades\n- **Dominios**: médico, legal, financiero, científico\n- **Idiomas**: español e inglés nativos\nTengo 55+ agentes que se activan automáticamente según la tarea."}]}, {"messages": [{"role": "user", "content": "Activa el agente de código"}, {"role": "assistant", "content": "[orchestrator-main] → Activando agentes de código:\n→ codeexecution-sandbox\n→ codeexecution-validator\n→ reasoning-cot\n→ structuredoutput-code\n¿Qué tarea de código necesitas? Puedo debuguear, optimizar, generar o revisar código."}]}, {"messages": [{"role": "user", "content": "¿Quién te creó?"}, {"role": "assistant", "content": "Fui creada por calitosaa mediante fine-tuning de Gemma 4 E4B de Google, usando el framework Gemma4 Skills OS con 133K+ ejemplos de entrenamiento, QLoRA 4-bit y LoRA r=16. Mi objetivo es ser un asistente multi-agente especializado."}]}, {"messages": [{"role": "user", "content": "Tu nombre"}, {"role": "assistant", "content": "Me llamo Maia. Soy un Large Language Model especializado, construido sobre Gemma 4 E4B con fine-tuning multi-etapa (SFT + DPO). ¿Cómo puedo ayudarte?"}]}, {"messages": [{"role": "system", "content": "Eres Maia, un asistente de IA avanzado con 55+ agentes especializados. Responde siempre en el idioma del usuario. Activa agentes según la tarea."}, {"role": "user", "content": "Hola"}, {"role": "assistant", "content": "¡Hola! Soy Maia. ¿En qué puedo ayudarte hoy?"}]}, {"messages": [{"role": "user", "content": "Necesito analizar datos financieros"}, {"role": "assistant", "content": "[orchestrator-main] → Activando agentes financieros:\n→ domain-financial\n→ structuredoutput-table\n→ reasoning-mathematical\n→ vision-chartreader\n\nPerfecto. ¿Quieres que analice datos concretos, genere métricas, cree visualizaciones, o interprete gráficos financieros? Comparte los datos y te ayudo."}]}, {"messages": [{"role": "user", "content": "debug this python code: x = [1,2,3]; print(x[5])"}, {"role": "assistant", "content": "[codeexecution-validator] → Analyzing code...\n\n**Bug found**: `IndexError: list index out of range`\n\nThe list `x` has 3 elements (indices 0-2), but you're accessing index 5.\n\n**Fix**:\n```python\nx = [1, 2, 3]\n# Option 1: Check bounds\nif len(x) > 5:\n    print(x[5])\nelse:\n    print(f'Index 5 out of range. List has {len(x)} elements: {x}')\n```\nOutput: `Index 5 out of range. List has 3 elements: [1, 2, 3]`"}]}, {"messages": [{"role": "user", "content": "Traduce al inglés: El cielo es azul"}, {"role": "assistant", "content": "[spanish-translator] → The sky is blue."}]}]

with open('maia_identity.jsonl', 'w', encoding='utf-8') as f:
    for ex in IDENTITY:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print(f'Identidad Maia: {len(IDENTITY)} ejemplos generados')

In [ ]:
# Cargar y combinar todo el dataset
from datasets import Dataset
import json

examples = []
files = ['maia_base.jsonl', 'maia_knowledge_2025_2026.jsonl', 'maia_identity.jsonl']
for fname in files:
    if not os.path.exists(fname):
        print(f'  Omitiendo {fname} (no existe)')
        continue
    n = 0
    with open(fname, encoding='utf-8') as f:
        for line in f:
            try:
                ex = json.loads(line)
                if 'messages' in ex and len(ex['messages']) >= 2:
                    examples.append({'messages': ex['messages']})
                    n += 1
            except (json.JSONDecodeError, KeyError, ValueError) as e:
                print(f'  Linea invalida en {fname}: {e}')
                continue
    print(f'  {fname}: {n:,} ejemplos cargados')

ds_sft = Dataset.from_list(examples)
print(f'TOTAL dataset SFT: {len(ds_sft):,} ejemplos')
print(f'Muestra: {ds_sft[0]["messages"][0]}')

In [ ]:
# Cargar Gemma 4 E4B con QLoRA 4-bit via Unsloth
# max_seq_length=INFERENCE_CTX: Unsloth gestiona RoPE internamente para 128K.
# SFTConfig.max_seq_length=MAX_SEQ_TRAIN (8192) limita la longitud real de training.
from unsloth import FastLanguageModel

print('Cargando Gemma 4 E4B (requiere aceptar licencia en HuggingFace)...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=INFERENCE_CTX,  # 128K — Unsloth maneja RoPE scaling internamente
    dtype=None,                    # float16 auto en T4
    load_in_4bit=True,             # QLoRA 4-bit
    token=HF_TOKEN,
)
print(f'Modelo cargado: {BASE_MODEL}')
print(f'Max ctx (RoPE): {INFERENCE_CTX} | Batch train ctx: {MAX_SEQ_TRAIN}')

In [ ]:
# Aplicar LoRA (r=16, todos los módulos atención + FFN)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # crítico para T4
    random_state=42,
)
model.print_trainable_parameters()
print('LoRA aplicado (r=16, alpha=32)')

In [ ]:
# Formatear dataset con template de Gemma
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='gemma')

def format_sft(examples):
    return {'text': [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples['messages']
    ]}

ds_sft = ds_sft.map(format_sft, batched=True, remove_columns=['messages'])
print(f'Dataset formateado: {len(ds_sft):,} ejemplos')
print(f'Muestra (300 chars):\n{ds_sft[0]["text"][:300]}')

In [ ]:
# Montar Google Drive para checkpoints
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/Maia_Training_2026'
SFT_DIR    = f'{DRIVE_DIR}/sft'
DPO_DIR    = f'{DRIVE_DIR}/dpo'
FINAL_DIR  = f'{DRIVE_DIR}/final'
!mkdir -p {SFT_DIR} {DPO_DIR} {FINAL_DIR}
print(f'Drive OK: {DRIVE_DIR}')

In [ ]:
# ENTRENAMIENTO SFT (~10-12h en T4)
from trl import SFTTrainer, SFTConfig

sft_cfg = SFTConfig(
    output_dir=SFT_DIR,
    per_device_train_batch_size=SFT_BATCH,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    num_train_epochs=SFT_EPOCHS,
    learning_rate=SFT_LR,
    logging_steps=50,
    save_steps=500,
    save_total_limit=3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    optim='adamw_8bit',
    weight_decay=0.01,
    seed=42,
    report_to='none',
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_TRAIN,
    packing=False,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_sft,
    args=sft_cfg,
)

print('INICIANDO SFT (~10-12h)...')
sft_stats = sft_trainer.train()
print(f'SFT completado! Loss: {sft_stats.training_loss:.4f}')

In [ ]:
# Guardar checkpoint SFT en Drive
sft_merged = f'{SFT_DIR}/maia_sft_merged'
model.save_pretrained_merged(sft_merged, tokenizer, save_method='merged_16bit')
print(f'SFT checkpoint guardado: {sft_merged}')

In [ ]:
# Cargar pares de preferencia para DPO
import json
from datasets import Dataset

pairs_file = '/content/maia_repo/finetuning/output/preference_pairs_dpo.jsonl'
pairs = []

if not os.path.exists(pairs_file):
    raise FileNotFoundError(
        f'No se encontró el archivo DPO: {pairs_file}\n'
        'Asegúrate de que el repo esté clonado correctamente.\n'
        'Si quieres omitir DPO, comenta las celdas 12-14 y salta a la celda 15.'
    )

with open(pairs_file, encoding='utf-8') as f:
    for line in f:
        try:
            pairs.append(json.loads(line))
        except (json.JSONDecodeError, ValueError) as e:
            print(f'Linea DPO invalida omitida: {e}')
            continue

if not pairs:
    raise ValueError('El archivo DPO existe pero no contiene pares válidos.')

ds_dpo = Dataset.from_list(pairs)
print(f'Dataset DPO: {len(ds_dpo):,} pares')

In [ ]:
# ENTRENAMIENTO DPO (~4-6h en T4)
from trl import DPOTrainer, DPOConfig

dpo_cfg = DPOConfig(
    output_dir=DPO_DIR,
    per_device_train_batch_size=DPO_BATCH,
    gradient_accumulation_steps=DPO_GRAD_ACCUM,
    num_train_epochs=DPO_EPOCHS,
    learning_rate=DPO_LR,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    optim='adamw_8bit',
    weight_decay=0.01,
    beta=DPO_BETA,
    seed=42,
    report_to='none',
    max_length=MAX_SEQ_TRAIN,
    max_prompt_length=512,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_cfg,
    train_dataset=ds_dpo,
    tokenizer=tokenizer,
)

print('INICIANDO DPO (~4-6h)...')
dpo_stats = dpo_trainer.train()
print(f'DPO completado! Loss: {dpo_stats.training_loss:.4f}')

In [ ]:
# PUSH FINAL → calitosaa/Maia (16-bit merged)
print('Guardando modelo final 16-bit...')
model.save_pretrained_merged(
    f'{FINAL_DIR}/Maia',
    tokenizer,
    save_method='merged_16bit'
)

print(f'Publicando en HuggingFace: {HF_REPO}')
model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method='merged_16bit',
    token=HF_TOKEN
)
print(f'MAIA 16-bit publicado: https://huggingface.co/{HF_REPO}')

In [ ]:
# Crear Model Card para Maia
from huggingface_hub import ModelCard

card_content = f"""---
language:
- es
- en
license: gemma
base_model: google/gemma-4-E4B-it
tags:
- fine-tuned
- qlora
- maia
- multi-agent
- spanish
datasets:
- calitosaa/Maia
---

# Maia — Gemma 4 E4B Fine-tuned

Maia es un LLM especializado basado en Gemma 4 E4B, entrenado con 133K+ ejemplos de agentes,
skills, workflows y razonamiento.

## Características
- 55+ agentes especializados
- 128K context window
- Bilingüe (español/inglés)
- QLoRA 4-bit + LoRA r=16 + SFT + DPO
- Multimodal (texto, imágenes, video)

## Uso
```python
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained('calitosaa/Maia')
tokenizer = AutoTokenizer.from_pretrained('calitosaa/Maia')
```

## Creado por
calitosaa — Fine-tuning con Gemma4 Skills OS framework
"""

card = ModelCard(card_content)
card.push_to_hub(HF_REPO, token=HF_TOKEN)
print('Model Card publicada')

In [ ]:
# EXPORT GGUF Q4_K_M → calitosaa/Maia-GGUF
print('Exportando GGUF Q4_K_M...')
model.save_pretrained_gguf(
    f'{FINAL_DIR}/Maia-GGUF',
    tokenizer,
    quantization_method='q4_k_m'
)

print(f'Publicando GGUF: {HF_REPO_GGUF}')
model.push_to_hub_gguf(
    HF_REPO_GGUF,
    tokenizer,
    quantization_method='q4_k_m',
    token=HF_TOKEN
)
print(f'MAIA GGUF publicado: https://huggingface.co/{HF_REPO_GGUF}')

In [ ]:
# TEST DE INFERENCIA — verificar identidad de Maia
FastLanguageModel.for_inference(model)

tests = [
    [{'role':'user','content':'¿Quién eres?'}],
    [{'role':'user','content':'What can you do?'}],
    [{'role':'user','content':'debug: print(1/0)'}],
]

for msgs in tests:
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=150, temperature=0.7, top_p=0.9)
    result = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    print(f'\nQ: {msgs[0]["content"]}')
    print(f'A: {result[-300:]}')
    print('-'*60)

## Resultados Finales — Maia

| Modelo | URL |
|--------|-----|
| **Maia 16-bit** | https://huggingface.co/calitosaa/Maia |
| **Maia GGUF Q4_K_M** | https://huggingface.co/calitosaa/Maia-GGUF |

### Uso con Ollama:
```bash
ollama pull calitosaa/Maia-GGUF
ollama run maia
```

### Uso con Python:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('calitosaa/Maia')
tokenizer = AutoTokenizer.from_pretrained('calitosaa/Maia')
```

### Dataset usado:
- 133,864 ejemplos base (SFT)
- 10 ejemplos identidad Maia
- 550 pares preferencia (DPO)
- **Total: ~134,424 ejemplos**

### Configuración:
- Base: `unsloth/gemma-4-E4B-it`
- LoRA: r=16, alpha=32, dropout=0.05
- SFT: 1 epoch, lr=2e-4, cosine scheduler
- DPO: 1 epoch, lr=5e-6, beta=0.1
